# 01 - Train and Save a Taxi Fare H2O Binary Model

This self-contained notebook uses the checked-in yellow taxi CSV to train a small H2O-3 GBM regression model, save it in H2O's binary model format, reload it, and verify prediction parity.

> Binary models must be loaded with the same H2O version used to save them. This notebook follows the H2O 3.46.0.12 `save_model()` and `load_model()` workflow.

## Setup

Select the repository `.venv` kernel, then install the checked-in notebook dependencies from the repository root:

```text
python -m pip install -r notebooks/requirements.txt
```

Copy `.env.example` to `.env` and fill in the Azure ML values for your environment. The notebook loads this file from the repository root, while existing process environment variables take precedence. Git ignores `.env`; do not commit or distribute it.

Keep `REGISTER_IN_AZURE=false` for the local save/load demonstration. Set it to `true` only when you want the optional registration cell to publish the model to your Azure ML workspace. Authentication uses `DefaultAzureCredential`, so use managed identity or sign in with the Azure CLI instead of storing credentials in the notebook.

This demo uses `h2o==3.46.0.12`. H2O starts its local server internally; no MOJO runtime JAR or Java command is used directly by the notebook.

In [32]:
from pathlib import Path
import hashlib
import json
import os
import shutil
import sys

import numpy as np
import pandas as pd
from dotenv import load_dotenv

for folder in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (folder / "data" / "taxi-data" / "raw" / "yellowTaxiData.csv").is_file():
        REPO_ROOT = folder
        break
else:
    raise FileNotFoundError("Run this notebook from inside the MLOPs-AzureML repository")

ENV_FILE = REPO_ROOT / ".env"
load_dotenv(ENV_FILE)

# The local JDK is used by H2O internally; notebook code stays in the Python API.
if not shutil.which("java"):
    java_bins = sorted((Path.home() / ".jdk").glob("jdk-17*/bin"), reverse=True)
    if java_bins:
        os.environ["JAVA_HOME"] = str(java_bins[0].parent)
        os.environ["PATH"] = str(java_bins[0]) + os.pathsep + os.environ["PATH"]

DATA_PATH = REPO_ROOT / "data" / "taxi-data" / "raw" / "yellowTaxiData.csv"
OUTPUT_DIR = REPO_ROOT / "tmp" / "h2o_binary" / "taxi_fare"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FEATURES = ["vendorID", "passengerCount", "tripDistance", "paymentType", "pickupHour"]
CATEGORICAL_FEATURES = ["vendorID", "paymentType"]
TARGET = "fareAmount"
H2O_VERSION = "3.46.0.12"
SEED = 42

print(f"Python: {sys.version.split()[0]}")
print(f"Data:   {DATA_PATH.relative_to(REPO_ROOT)}")
print(f"Output: {OUTPUT_DIR.relative_to(REPO_ROOT)}")

Python: 3.12.10
Data:   data\taxi-data\raw\yellowTaxiData.csv
Output: tmp\h2o_binary\taxi_fare


In [33]:
taxi = pd.read_csv(DATA_PATH)
taxi["pickupHour"] = pd.to_datetime(taxi["tpepPickupDateTime"]).dt.hour
model_data = taxi.loc[
    (taxi["tripDistance"] > 0) & (taxi[TARGET] > 0),
    [*FEATURES, TARGET],
].dropna()

print(f"Training rows: {len(model_data):,}")
display(model_data.head())
display(model_data[["tripDistance", TARGET]].describe())

Training rows: 4,964


,vendorID,passengerCount,tripDistance,paymentType,pickupHour,fareAmount
0,2,1,2.09,1,12,10.5
1,1,3,1.50,2,17,8.5
2,1,1,1.80,1,7,8.5
3,2,1,1.96,1,0,8.0
4,2,1,3.60,1,23,13.5


,tripDistance,fareAmount
count,4964.000000,4964.000000
mean,2.899654,12.451966
std,3.591882,10.604197
min,0.050000,0.010000
25%,1.000000,6.500000
50%,1.690000,9.000000
75%,3.000000,14.000000
max,38.580000,132.000000


## Train a Small H2O GBM

The model predicts `fareAmount` from five easy-to-explain taxi fields and is intentionally small so the demo stays focused on the binary-model save/load lifecycle.

In [34]:
import h2o
from h2o.estimators.gbm import H2OGradientBoostingEstimator

if h2o.__version__ != H2O_VERSION:
    raise RuntimeError(f"Expected h2o=={H2O_VERSION}, found {h2o.__version__}")

h2o.init(max_mem_size="2G", nthreads=-1)
h2o_data = h2o.H2OFrame(model_data)
for column in CATEGORICAL_FEATURES:
    h2o_data[column] = h2o_data[column].asfactor()

train, validation = h2o_data.split_frame(ratios=[0.8], seed=SEED)
model = H2OGradientBoostingEstimator(
    model_id="taxi-fare-gbm",
    ntrees=60,
    max_depth=5,
    learn_rate=0.08,
    seed=SEED,
)
model.train(x=FEATURES, y=TARGET, training_frame=train, validation_frame=validation)

performance = model.model_performance(validation)
print({"rmse": performance.rmse(), "mae": performance.mae(), "r2": performance.r2()})

Checking whether there is an H2O instance running at http://localhost:54321..... not found.
Attempting to start a local H2O server...
; OpenJDK 64-Bit Server VM Microsoft-11926163 (build 17.0.16+8-LTS, mixed mode, sharing)
  Starting server from C:\Users\jomedin\Documents\MLOPs-AzureML\.venv\Lib\site-packages\h2o\backend\bin\h2o.jar
  Ice root: C:\Users\jomedin\AppData\Local\Temp\tmpxo2o6jx1
  JVM stdout: C:\Users\jomedin\AppData\Local\Temp\tmpxo2o6jx1\h2o_jomedin_started_from_python.out
  JVM stderr: C:\Users\jomedin\AppData\Local\Temp\tmpxo2o6jx1\h2o_jomedin_started_from_python.err
  Server is running at http://127.0.0.1:54321
Connecting to H2O server at http://127.0.0.1:54321 ... successful.


H2O_cluster_uptime:,03 secs
H2O_cluster_timezone:,America/New_York
H2O_data_parsing_timezone:,UTC
H2O_cluster_version:,3.46.0.12
H2O_cluster_version_age:,30 days
H2O_cluster_name:,H2O_from_python_jomedin_mjzyxn
H2O_cluster_total_nodes:,1
H2O_cluster_free_memory:,2 Gb
H2O_cluster_total_cores:,8
H2O_cluster_allowed_cores:,8
H2O_cluster_status:,"locked, healthy"



+------------------------------------------------------------------+
| You are running the community edition of H2O-3 OSS.              |
|                                                                  |
| For commercial use, H2O-3 Secure is now recommended.             |
| This includes production support, CVE fixes, multi-node scaling, |
| model artifact extraction, and more.                             |
| See h2o.ai/h2o-3/oss-vs-secure for additional details.           |
| Contact enterprise@h2o.ai to upgrade.                            |
+------------------------------------------------------------------+
Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
gbm Model Build progress: |██████████████████████████████████████████████████████| (done) 100%
{'rmse': 2.9709854940143163, 'mae': 1.5734441920470132, 'r2': 0.911051504780502}


## Save the Binary Model and Golden Test Files

`h2o.save_model()` writes the version-specific H2O binary model. The output folder also contains a small input sample, expected predictions, and a checksum manifest for Notebook 02.

In [35]:
model_path = Path(
    h2o.save_model(
        model=model,
        path=str(OUTPUT_DIR),
        filename="taxi-fare-gbm",
        force=True,
    )
)

# Keep a few validation rows as a versioned scoring fixture.
golden = validation.as_data_frame().head(20)
golden_input_path = OUTPUT_DIR / "golden_input.csv"
golden_expected_path = OUTPUT_DIR / "golden_expected.csv"
golden[FEATURES].to_csv(golden_input_path, index=False)

expected_h2o = h2o.H2OFrame(golden[FEATURES])
for column in CATEGORICAL_FEATURES:
    expected_h2o[column] = expected_h2o[column].asfactor()
model.predict(expected_h2o).as_data_frame().to_csv(golden_expected_path, index=False)

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

manifest = {
    "model_name": "taxi-fare-h2o-binary",
    "model_version": "1",
    "model_format": "h2o_binary",
    "h2o_version": h2o.__version__,
    "model_file": model_path.name,
    "target": TARGET,
    "features": FEATURES,
    "categorical_features": CATEGORICAL_FEATURES,
    "files": {
        model_path.name: sha256(model_path),
        golden_input_path.name: sha256(golden_input_path),
        golden_expected_path.name: sha256(golden_expected_path),
    },
}
manifest_path = OUTPUT_DIR / "model_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")

print(f"Binary model: {model_path.relative_to(REPO_ROOT)}")
print(f"Manifest:     {manifest_path.relative_to(REPO_ROOT)}")

Parse progress: |

c:\Users\jomedin\Documents\MLOPs-AzureML\.venv\Lib\site-packages\h2o\frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"


████████████████████████████████████████████████████████████████| (done) 100%
gbm prediction progress: |███████████████████████████████████████████████████████| (done) 100%
Binary model: tmp\h2o_binary\taxi_fare\taxi-fare-gbm
Manifest:     tmp\h2o_binary\taxi_fare\model_manifest.json


c:\Users\jomedin\Documents\MLOPs-AzureML\.venv\Lib\site-packages\h2o\frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"


## Reload and Verify the Binary Model

`h2o.load_model()` reloads the saved model into the running H2O cluster. The reloaded model must reproduce the original model's golden predictions.

In [36]:
loaded_model = h2o.load_model(str(model_path))

loaded_input = h2o.H2OFrame(golden[FEATURES])
for column in CATEGORICAL_FEATURES:
    loaded_input[column] = loaded_input[column].asfactor()

expected = pd.read_csv(golden_expected_path)
actual = loaded_model.predict(loaded_input).as_data_frame()
np.testing.assert_allclose(expected["predict"], actual["predict"], rtol=1e-6, atol=1e-6)

display(pd.DataFrame({"original": expected["predict"], "loaded": actual["predict"]}).head(10))
print("The loaded binary model matches the original H2O model.")

Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
gbm prediction progress: |███████████████████████████████████████████████████████| (done) 100%


c:\Users\jomedin\Documents\MLOPs-AzureML\.venv\Lib\site-packages\h2o\frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"


,original,loaded
0,34.823667,34.823667
1,20.211054,20.211054
2,13.282541,13.282541
3,9.505314,9.505314
4,7.076866,7.076866
5,18.151701,18.151701
6,11.130871,11.130871
7,6.696699,6.696699
8,7.701867,7.701867
9,12.329862,12.329862


The loaded binary model matches the original H2O model.


In [37]:
def env_flag(name, default=False):
    value = os.getenv(name)
    if value is None:
        return default

    normalized = value.strip().lower()
    if normalized not in {"1", "0", "true", "false", "yes", "no", "on", "off"}:
        raise ValueError(f"{name} must be true or false")
    return normalized in {"1", "true", "yes", "on"}


REGISTER_IN_AZURE = env_flag("REGISTER_IN_AZURE")
SUBSCRIPTION_ID = os.getenv("AZURE_SUBSCRIPTION_ID", "").strip()
RESOURCE_GROUP = os.getenv("AZURE_RESOURCE_GROUP", "").strip()
WORKSPACE_NAME = os.getenv("AZUREML_WORKSPACE_NAME", "").strip()

if REGISTER_IN_AZURE:
    azure_settings = {
        "AZURE_SUBSCRIPTION_ID": SUBSCRIPTION_ID,
        "AZURE_RESOURCE_GROUP": RESOURCE_GROUP,
        "AZUREML_WORKSPACE_NAME": WORKSPACE_NAME,
    }
    missing_settings = [name for name, value in azure_settings.items() if not value]
    if missing_settings:
        raise ValueError(
            f"Set {', '.join(missing_settings)} in {ENV_FILE.name} or the process environment"
        )

    from azure.ai.ml import MLClient
    from azure.ai.ml.constants import AssetTypes
    from azure.ai.ml.entities import Model
    from azure.identity import DefaultAzureCredential

    ml_client = MLClient(
        DefaultAzureCredential(exclude_interactive_browser_credential=False),
        SUBSCRIPTION_ID,
        RESOURCE_GROUP,
        WORKSPACE_NAME,
    )
    registered_model = ml_client.models.create_or_update(
        Model(
            path=str(OUTPUT_DIR),
            name=manifest["model_name"],
            version=manifest["model_version"],
            type=AssetTypes.CUSTOM_MODEL,
            description="H2O 3.46.0.12 binary taxi fare model",
            tags={
                "model_format": manifest["model_format"],
                "h2o_version": manifest["h2o_version"],
                "model_sha256": manifest["files"][model_path.name],
            },
        )
    )
    print(f"Registered model: {registered_model.name}:{registered_model.version}")
else:
    print("Azure registration skipped. Binary model save/load validation is complete.")

Azure registration skipped. Binary model save/load validation is complete.


## Outputs

Notebook 02 consumes this directory exactly as written. Keep the binary model and manifest together, and pin its Azure ML runtime to `h2o==3.46.0.12`.

Before distributing the notebook, clear all cell outputs. H2O startup logs can contain local usernames, temporary directories, and absolute paths even though the notebook source is portable.

In [38]:
artifact_paths = [model_path, golden_input_path, golden_expected_path, manifest_path]
artifact_table = pd.DataFrame(
    [
        {"file": path.name, "bytes": path.stat().st_size, "sha256": sha256(path)}
        for path in artifact_paths
    ]
)
display(artifact_table)

# Release the local H2O server when preparation is complete.
h2o.cluster().shutdown(prompt=False)

,file,bytes,sha256
0,taxi-fare-gbm,140031,995303cdb3e1462cae499693cca47ffc3d23b63a43ea35...
1,golden_input.csv,345,8c44d35212bd127d08a689e536840cb80e80ccdf1ef9b2...
2,golden_expected.csv,393,4832dae196d92885c9ed0af739ace089aaf286cb375e82...
3,model_manifest.json,687,0cd71ebccd9ffff35bae298cf0b938ff5ec3edad5ccaae...


H2O session _sid_abf6 closed.
